# NHL Data Collection

This notebook collects play-by-play data from the NHL API for the **past 5 regular seasons (2019–2023)**.

## Overview
- Generates game IDs for all regular season games across multiple seasons
- Fetches play-by-play data from the NHL API
- Saves raw data to a parquet file for later processing


## Imports


In [1]:
import pandas as pd
import os
from tqdm.notebook import tqdm
import sys

# Add src to path for imports
sys.path.append('../../')
from src.data.collectors.nhl_api import get_game_data, generate_game_ids


## Constants and Configuration


In [2]:
# Configuration
# Collect past 5 regular seasons (2019–2023)
SEASONS = ["2019", "2020", "2021", "2022", "2023"]
GAME_TYPE = "02"  # 01 = Preseason, 02 = Regular Season, 03 = Playoffs
NUM_GAMES = 1312  # Expected number of regular-season games per season in recent years
DATA_FILE_NAME = "../../data/raw/nhl_raw_plays_2019_2023.parquet"  # Save to data/raw directory

# Generate all game IDs across seasons
game_ids = []
for season in SEASONS:
    season_ids = generate_game_ids(season=season, game_type=GAME_TYPE, num_games=NUM_GAMES)
    print(f"Season {season}: generated {len(season_ids)} game IDs.")
    print(f"  First ID: {season_ids[0]}, Last ID: {season_ids[-1]}")
    game_ids.extend(season_ids)

print(f"\nTotal game IDs across seasons {SEASONS[0]}–{SEASONS[-1]}: {len(game_ids)}")
print(f"First ID: {game_ids[0]}, Last ID: {game_ids[-1]}")


Season 2019: generated 1312 game IDs.
  First ID: 2019020001, Last ID: 2019021312
Season 2020: generated 1312 game IDs.
  First ID: 2020020001, Last ID: 2020021312
Season 2021: generated 1312 game IDs.
  First ID: 2021020001, Last ID: 2021021312
Season 2022: generated 1312 game IDs.
  First ID: 2022020001, Last ID: 2022021312
Season 2023: generated 1312 game IDs.
  First ID: 2023020001, Last ID: 2023021312

Total game IDs across seasons 2019–2023: 6560
First ID: 2019020001, Last ID: 2023021312


## Data Collection


In [3]:
all_plays_data = []

# Check if data file already exists
if not os.path.exists(DATA_FILE_NAME):
    print(f"Data file {DATA_FILE_NAME} does not exist. Starting data collection...")
    
    # Create data/raw directory if it doesn't exist
    os.makedirs(os.path.dirname(DATA_FILE_NAME), exist_ok=True)
    
    # Loop through all game IDs with progress bar
    for game_id in tqdm(game_ids, desc="Fetching data"):
        plays = get_game_data(game_id)
        
        # Add plays to all_plays_data
        if plays:
            all_plays_data.extend(plays)
    
    print(f"\nTotal plays collected: {len(all_plays_data)}")
    
    # Convert to DataFrame
    df = pd.DataFrame(all_plays_data)
    
    # Save to Parquet file
    df.to_parquet(DATA_FILE_NAME, index=False)
    print(f"Data saved to {DATA_FILE_NAME}")
else:
    print(f"Loading data from {DATA_FILE_NAME}...")
    df = pd.read_parquet(DATA_FILE_NAME)
    print(f"Loaded {len(df)} plays from {DATA_FILE_NAME}")

# Display first few rows of the DataFrame
print(f"\nDataFrame shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
df.head()


Data file ../../data/raw/nhl_raw_plays_2019_2023.parquet does not exist. Starting data collection...


Fetching data:   0%|          | 0/6560 [00:00<?, ?it/s]

Error fetching data for game 2019021083: 404 Client Error: Not Found for url: https://api-web.nhle.com/v1/gamecenter/2019021083/play-by-play
Error fetching data for game 2019021084: 404 Client Error: Not Found for url: https://api-web.nhle.com/v1/gamecenter/2019021084/play-by-play
Error fetching data for game 2019021085: 404 Client Error: Not Found for url: https://api-web.nhle.com/v1/gamecenter/2019021085/play-by-play
Error fetching data for game 2019021086: 404 Client Error: Not Found for url: https://api-web.nhle.com/v1/gamecenter/2019021086/play-by-play
Error fetching data for game 2019021087: 404 Client Error: Not Found for url: https://api-web.nhle.com/v1/gamecenter/2019021087/play-by-play
Error fetching data for game 2019021088: 404 Client Error: Not Found for url: https://api-web.nhle.com/v1/gamecenter/2019021088/play-by-play
Error fetching data for game 2019021089: 404 Client Error: Not Found for url: https://api-web.nhle.com/v1/gamecenter/2019021089/play-by-play
Error fetchin

,eventId,periodDescriptor,timeInPeriod,timeRemaining,situationCode,homeTeamDefendingSide,typeCode,typeDescKey,sortOrder,details,pptReplayUrl
0,8,"{'number': 1, 'periodType': 'REG', 'maxRegulat...",00:00,20:00,1551,right,520,period-start,8,NaN,NaN
1,9,"{'number': 1, 'periodType': 'REG', 'maxRegulat...",00:00,20:00,1551,right,502,faceoff,10,"{'eventOwnerTeamId': 10, 'losingPlayerId': 847...",NaN
2,10,"{'number': 1, 'periodType': 'REG', 'maxRegulat...",00:25,19:35,1551,right,505,goal,11,"{'xCoord': 85, 'yCoord': -1, 'zoneCode': 'O', ...",NaN
3,11,"{'number': 1, 'periodType': 'REG', 'maxRegulat...",00:25,19:35,1551,right,502,faceoff,14,"{'eventOwnerTeamId': 9, 'losingPlayerId': 8477...",NaN
4,12,"{'number': 1, 'periodType': 'REG', 'maxRegulat...",00:38,19:22,1551,right,507,missed-shot,15,"{'xCoord': 28, 'yCoord': -37, 'zoneCode': 'O',...",NaN
